# Retrain the Fallacy Detector (v3 data, accuracy-tuned)

Trains **both stages** on the improved data and exports drop-in models.

### What changed vs. the previous version of this notebook

| | before | now |
|---|---|---|
| detector positives | 930 (contrastive + MAFALDA only) | **~2,780** — `edu_train.csv`'s 1,849 labelled fallacies were being fed to the typer *only*, so the detector never saw the 110 false-dilemma examples. That is why it rated the textbook false dilemma **0.41**. |
| best-epoch metric | macro-F1 with a denominator that changed every epoch | fixed over all classes, so epochs are actually comparable |
| detector threshold | hardcoded `0.70`, never measured | **swept on the val set** and shipped inside `classes.json` |
| encoder | distilbert, one seed | distilbert **and** distilroberta × 2 seeds — the val set picks the winner |
| optimiser | flat LR, no clipping | linear warmup + decay, gradient clipping |
| `intentional` class | 112 training examples, 0 in the real-world eval — could only ever be a *wrong* prediction | dropped (13 classes) |
| speed | pad-to-96, fp32 | dynamic padding + fp16 — roughly 3× faster, which pays for the wider sweep |

Nothing is selected on the test set: the document split is train / val / test, val picks
the epoch, encoder, seed and threshold, and test is scored once at the end.

---

### What to do
1. **Runtime → Change runtime type → T4 GPU**
2. Upload your **`data/`** folder into the Colab session (file pane on the left).
   It needs: `contrastive_dataset.csv`, `edu_train.csv`, `gold_standard_dataset.jsonl`.
3. **Runtime → Run all** (~15–25 min: 8 fine-tunes, 4 per stage)
4. The last cells zip the models and download them. Unzip into
   `fallacysuspect/models/two_stage/` (replacing the old `*_distilbert/` folders).

## 0. Setup

In [ ]:
%pip install -q transformers torch scikit-learn pandas numpy

In [ ]:
import json, os, glob, time, copy
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type != "cuda":
    print("  !! No GPU — Runtime > Change runtime type > T4 GPU (CPU works but is slow)")

# --- find the uploaded data/ folder in the Colab session ---
MARKERS = ["contrastive_dataset.csv", "gold_standard_dataset.jsonl",
           "edu_all.csv", "climate_all.csv"]
def has_data(d):
    return bool(d) and all(os.path.exists(os.path.join(d, m)) for m in MARKERS)

DATA_DIR = None
for c in ["data", "/content/data", "/content", ".", "../data"]:
    if has_data(c):
        DATA_DIR = c; break
if DATA_DIR is None:                      # last resort: search the session
    hits = glob.glob(os.path.join("/content", "**", MARKERS[0]), recursive=True)
    if hits:
        DATA_DIR = os.path.dirname(hits[0])

print("DATA_DIR:", DATA_DIR, "| found:", has_data(DATA_DIR))
assert has_data(DATA_DIR), (
    "Upload the whole data/ folder into the session. Needed: " + ", ".join(MARKERS))

## 1. Config

In [ ]:
# ---- what to train -------------------------------------------------------
# Two encoders in the same size/speed class. distilroberta is usually a little
# stronger than distilbert on argument-structure tasks; we train both and keep
# whichever wins on VAL. The saved folder is still named *_distilbert either way,
# because the app looks for that name.
MODEL_CANDIDATES = ["distilbert-base-uncased", "distilroberta-base"]
SEEDS            = [42, 1337]   # the val set is small -> a single seed is noisy

MAX_LEN     = 96
BATCH       = 32
LR          = 3e-5
WARMUP_FRAC = 0.10        # linear warmup, then linear decay
CLIP_NORM   = 1.0
FP16        = True        # ~2.5x faster on a T4; ignored on CPU
DET_EPOCHS  = 6
TYP_EPOCHS  = 8           # 13 classes, needs more

TEST_FRAC   = 0.20        # MAFALDA *documents* held out for the final score
VAL_FRAC    = 0.15        # ...and for picking the best epoch/encoder/seed
MIN_WORDS   = 5

# --------------------------------------------------------------------------
# DATA. The three source projects don't agree with each other, and squeezing
# them together is where most of the remaining accuracy is. Each switch below
# is one inconsistency being reconciled.
# --------------------------------------------------------------------------

# (a) edu_train.csv is only 1,849 of LOGIC's 2,452 rows — edu_dev/edu_test were
#     never touched. We evaluate on held-out MAFALDA *documents*, a different
#     corpus entirely, so LOGIC's own dev/test splits are just training data to
#     us. No leakage. Uses edu_all.csv instead. (+603 rows)
USE_EDU_ALL = True

# (b) climate_all.csv — 1,351 rows from the same causalNLP project, same 13-class
#     taxonomy, zero text overlap with edu, and NEVER USED. It's climate/energy
#     debate prose, which is the closest domain we have to the transcripts the app
#     actually gets. (+1,009 after dropping 'intentional')
USE_CLIMATE = True

# (c) LOGIC has 143 slippery-slope examples, but when causalNLP collapsed their
#     185 original labels into 13 they folded EVERY ONE into 'faulty
#     generalization' — so "if we use one more can of hairspray, earth will no
#     longer exist" currently trains as a faulty generalization. That both starves
#     slippery slope (19 examples, all from MAFALDA) and poisons the largest class.
#     The original label survives in edu_all's `old_label` column. Recover it.
RECOVER_SLIPPERY_SLOPE = True

# (d) edu_train.csv's 1,849 labelled fallacies were fed to the TYPER ONLY, so the
#     detector trained on ~930 positives and never saw the 110 false dilemmas --
#     which is why it rates the textbook false dilemma 0.41.
USE_EDU_AS_DETECTOR_POSITIVES = True

# (e) 'intentional' (485 rows across edu+climate) has ZERO examples in MAFALDA, so
#     on the real-world eval it can never be a correct prediction, only a wrong one.
#     In the app it surfaces as the useless warning "uses a deliberate rhetorical
#     trick". 'miscellaneous' (3 rows) is a junk drawer. Drop both.
DROP_CLASSES = {"intentional", "miscellaneous"}

# NOT used: user_study_examples_with_labels.jsonl — all 20 of its documents are
# already inside gold_standard_dataset.jsonl, so adding it would leak test docs
# into training. non_annotated_dataset.jsonl has 9,545 rows but every `labels`
# field is empty — no supervision to be had.

DATA_VERSION = 4          # v4 = all three projects reconciled + tuned threshold
OUT_DIR = "/content/models/two_stage" if os.path.isdir("/content") else "models/two_stage"
os.makedirs(OUT_DIR, exist_ok=True)
print("models will be written to:", OUT_DIR)
print("candidates:", MODEL_CANDIDATES, "| seeds:", SEEDS)

## 1b. Pre-download DistilBERT (with retries)

The base weights are ~268 MB from Hugging Face. Grabbing them up front — with retry +
resume — means a flaky download can't masquerade as "training is stuck". Normally takes
seconds; if a run stalls, just re-run **this cell** (it resumes from cache).

In [ ]:
# Plain from_pretrained in a retry loop — no huggingface_hub kwargs (they change
# between versions; `resume_download` is deprecated/removed and would crash here).
# Downloads resume from cache automatically on retry.
for name in MODEL_CANDIDATES:
    for attempt in range(1, 6):
        try:
            t0 = time.time()
            AutoTokenizer.from_pretrained(name)
            AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)
            print(f"cached {name} in {time.time()-t0:.0f}s")
            break
        except Exception as exc:
            print(f"  {name} attempt {attempt} failed ({type(exc).__name__}: {exc}) — retrying in 5s...")
            time.sleep(5)
    else:
        raise RuntimeError(f"Could not download {name} after 5 tries. "
                           "Runtime > Restart session, then Run all.")
print("\nall base encoders cached — training will load instantly from here.")

## 2. Reconcile the three sources

The data comes from three projects that don't agree with each other:

| source | what it gives us | what it can't |
|---|---|---|
| **MAFALDA** (ChadiHelwe) | real-world text, labelled *sentence by sentence*, and the only source with explicit **non**-fallacious sentences | small — 200 documents |
| **LOGIC / causalNLP** — `edu_*` + `climate_*` | ~3,300 labelled fallacies over 13 types, incl. a whole climate/energy-debate domain | every row is a fallacy, so it can't teach "this is fine" |
| **Kaggle contrastive** | 703 matched fallacy/valid pairs — the same claim argued soundly and unsoundly | short, synthetic, no type labels |

Each is useless alone. MAFALDA supplies the negatives and the honest eval; LOGIC supplies
the volume and the types; the contrastive set supplies *argument-shaped* negatives, which is
the hard case — MAFALDA's negatives are mostly ordinary prose, so without the contrastive
pairs the detector learns "sounds like an argument → fallacy".

Reconciling them means three concrete fixes, all switched on in the config cell above:
LOGIC's `slippery slope` label is recovered from `old_label` (it had been collapsed into
`faulty generalization`), `intentional`/`miscellaneous` are dropped as unlearnable, and the
LOGIC fallacies are finally fed to the **detector** and not just the typer.

MAFALDA is split **by document**, and the cell asserts that no held-out sentence appears
anywhere in training.

In [ ]:
SPLIT_SEED = 42          # the document split is fixed — only the *training* seed varies

# MAFALDA's 23 fine-grained types -> our taxonomy (LOGIC's 13, minus the dropped
# junk classes, plus slippery slope which LOGIC collapsed away)
MAFALDA_TO_OURS = {
    "hasty generalization": "faulty generalization",
    "appeal to ridicule": "appeal to emotion",
    "slippery slope": "slippery slope",
    "causal oversimplification": "false causality",
    "ad hominem": "ad hominem",
    "false dilemma": "false dilemma",
    "appeal to fear": "appeal to emotion",
    "appeal to nature": "fallacy of relevance",
    "false analogy": "fallacy of logic",
    "ad populum": "ad populum",
    "false causality": "false causality",
    "straw man": "fallacy of extension",
    "guilt by association": "ad hominem",
    "appeal to (false) authority": "fallacy of credibility",
    "equivocation": "equivocation",
    "circular reasoning": "circular reasoning",
    "appeal to anger": "appeal to emotion",
    "appeal to worse problems": "fallacy of relevance",
    "appeal to tradition": "fallacy of relevance",
    "fallacy of division": "fallacy of logic",
    "appeal to positive emotion": "appeal to emotion",
    "tu quoque": "ad hominem",
    "appeal to pity": "appeal to emotion",
}


# =========================================================================
# SOURCE 1 — MAFALDA (real-world, sentence-level, INCLUDING negatives).
# The only source with 'nothing' sentences, so the only one that can teach the
# detector what ordinary prose looks like — and the only honest eval set.
# Split BY DOCUMENT so no sentence leaks across the split.
# =========================================================================
def load_mafalda_docs():
    docs = []
    for line in open(os.path.join(DATA_DIR, "gold_standard_dataset.jsonl"), encoding="utf-8"):
        if not line.strip():
            continue
        rec = json.loads(line)
        swl = rec["sentences_with_labels"]
        if isinstance(swl, str):
            swl = json.loads(swl)
        sents = []
        for sent, labels in swl.items():
            flat = [x for grp in labels for x in (grp if isinstance(grp, list) else [grp])]
            types = [MAFALDA_TO_OURS[t] for t in flat if t != "nothing" and t in MAFALDA_TO_OURS]
            types = [t for t in types if t not in DROP_CLASSES]
            sent = sent.strip()
            if len(sent.split()) >= MIN_WORDS:
                sents.append((sent, types))
        if sents:
            docs.append(sents)
    return docs

rng = np.random.RandomState(SPLIT_SEED)
docs = load_mafalda_docs()
order = rng.permutation(len(docs))
n_test, n_val = int(TEST_FRAC * len(docs)), int(VAL_FRAC * len(docs))
test_ids, val_ids, train_ids = order[:n_test], order[n_test:n_test+n_val], order[n_test+n_val:]

maf_train = [s for i in train_ids for s in docs[i]]
maf_val   = [s for i in val_ids   for s in docs[i]]
maf_test  = [s for i in test_ids  for s in docs[i]]
print(f"MAFALDA: {len(docs)} docs -> {len(maf_train)} train / {len(maf_val)} val / {len(maf_test)} test sentences")

# Everything below is filtered against this. MAFALDA and LOGIC share some source
# text, so a handful of held-out sentences appear verbatim in the LOGIC files —
# without this filter they leak into training and quietly inflate the test score.
HELD_OUT = {s for s, _ in maf_val} | {s for s, _ in maf_test}

def det_df(pairs):
    return pd.DataFrame([(s, "fallacy" if ty else "not_fallacy") for s, ty in pairs],
                        columns=["text", "label"])

def typ_df(pairs):
    return pd.DataFrame([(s, ty[0]) for s, ty in pairs if ty], columns=["text", "label"])


# =========================================================================
# SOURCE 2 — causalNLP / LOGIC: edu (general) + climate (energy debate).
# Every row is a fallacy; no negatives. Same 13-class taxonomy in both files.
# =========================================================================
fallacy_parts = []

edu_file = "edu_all.csv" if USE_EDU_ALL else "edu_train.csv"
edu = pd.read_csv(os.path.join(DATA_DIR, edu_file))
edu_txt = edu["source_article"].astype(str).str.strip()
edu_lab = edu["updated_label"].astype(str).str.strip()

if RECOVER_SLIPPERY_SLOPE and "old_label" in edu.columns:
    is_ss = edu["old_label"].astype(str).str.strip().str.lower() == "slippery slope"
    print(f"\nrecovering slippery slope: {int(is_ss.sum())} rows, currently mislabelled "
          f"{dict(edu_lab[is_ss].value_counts())}")
    edu_lab = edu_lab.mask(is_ss, "slippery slope")

fallacy_parts.append(pd.DataFrame({"text": edu_txt, "label": edu_lab, "src": edu_file}))

if USE_CLIMATE:
    cl = pd.read_csv(os.path.join(DATA_DIR, "climate_all.csv"))
    fallacy_parts.append(pd.DataFrame({
        "text": cl["source_article"].astype(str).str.strip(),
        "label": cl["logical_fallacies"].astype(str).str.strip(),
        "src": "climate_all.csv"}))

fal = pd.concat(fallacy_parts, ignore_index=True)
fal = fal[~fal["label"].isin(DROP_CLASSES)]
fal = fal[fal["text"].str.split().str.len() >= MIN_WORDS]
fal = fal.drop_duplicates(subset="text")

n_leak = int(fal["text"].isin(HELD_OUT).sum())
fal = fal[~fal["text"].isin(HELD_OUT)].reset_index(drop=True)
print(f"\ncurated fallacy pool: {len(fal)} rows "
      f"({n_leak} dropped — they duplicate a held-out MAFALDA sentence)")
print(fal["src"].value_counts().to_string())


# =========================================================================
# SOURCE 3 — Kaggle contrastive set: 703 fallacy / 703 valid, matched pairs.
# Its 'valid' half is the only ARGUMENT-shaped negative we have (MAFALDA's
# negatives are mostly plain prose), so it teaches the detector the hard
# distinction: the same claim, argued soundly vs unsoundly.
# =========================================================================
con = pd.read_csv(os.path.join(DATA_DIR, "contrastive_dataset.csv"))
con_df = pd.DataFrame({"text": con["text"].astype(str).str.strip(),
                       "label": np.where(con["label"] == "fallacy", "fallacy", "not_fallacy")})
con_df = con_df[~con_df["text"].isin(HELD_OUT)]


# ---- Stage 1: detector -----------------------------------------------------
det_parts = [con_df, det_df(maf_train)]
if USE_EDU_AS_DETECTOR_POSITIVES:
    det_parts.append(pd.DataFrame({"text": fal["text"], "label": "fallacy"}))

det_train = pd.concat(det_parts).drop_duplicates(subset="text").reset_index(drop=True)
det_val, det_test = det_df(maf_val), det_df(maf_test)

# ---- Stage 2: typer --------------------------------------------------------
typ_train = pd.concat([fal[["text", "label"]], typ_df(maf_train)]) \
              .drop_duplicates(subset="text").reset_index(drop=True)
typ_val, typ_test = typ_df(maf_val), typ_df(maf_test)

print(f"\ndetector: {len(det_train)} train / {len(det_val)} val / {len(det_test)} test")
print(det_train["label"].value_counts().to_string())
pos = int((det_train["label"] == "fallacy").sum())
neg = int((det_train["label"] == "not_fallacy").sum())
print(f"  positives:negatives = {pos/max(neg,1):.1f}:1 — class weights and the tuned")
print( "  threshold correct for this, but watch the false-alarm rate on TEST below.")

print(f"\ntyper: {len(typ_train)} train / {len(typ_val)} val / {len(typ_test)} test"
      f" | {typ_train['label'].nunique()} classes")
print(typ_train["label"].value_counts().to_string())

# The typer's val/test sets are tiny (a few dozen fallacies over ~10 classes), so its
# macro-F1 carries a real error bar. Say so, so the number doesn't get over-read.
print(f"\n!! typer val has only {len(typ_val)} sentences over {typ_val['label'].nunique()} classes"
      f" and test only {len(typ_test)} over {typ_test['label'].nunique()}"
      " — treat its macro-F1 as +/- 0.1, not a precise figure.")

leaked = HELD_OUT & (set(typ_train["text"]) | set(det_train["text"]))
assert not leaked, f"LEAK: {len(leaked)} held-out sentences found in training"
assert len(det_val) and len(det_test) and len(typ_val) and len(typ_test), "a split came out empty"
print("\nno leakage: held-out MAFALDA sentences appear nowhere in training.")

## 3. Fine-tuning

For each stage we train every `(encoder, seed)` combination, score each epoch on the
**val** set, and keep the single best checkpoint. Class weights handle the rare classes;
warmup + decay and gradient clipping keep the small-data fine-tune stable.

Only after the winner is locked in do we touch the **test** set — that number is the honest
one. For the detector we also sweep the decision threshold on val, because the app gates on
`P(fallacy) >= threshold` rather than taking an argmax, and the 0.70 in `config.py` was a
guess.

In [ ]:
USE_AMP = FP16 and DEVICE.type == "cuda"


class DS(Dataset):
    """Holds raw text — tokenisation happens per batch so we pad to the longest
    sentence in the batch, not to MAX_LEN. Most sentences are far shorter than 96
    tokens, so this alone roughly halves training time."""
    def __init__(self, texts, labels):
        self.texts, self.y = list(texts), list(labels)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i):  return self.texts[i], self.y[i]


def _collate(tok):
    def fn(batch):
        enc = tok([b[0] for b in batch], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt")
        enc.pop("token_type_ids", None)
        enc["labels"] = torch.tensor([b[1] for b in batch], dtype=torch.long)
        return enc
    return fn


def _loader(df, cmap, tok, shuffle=False):
    return DataLoader(DS(df["text"], df["label"].map(cmap)), batch_size=BATCH,
                      shuffle=shuffle, collate_fn=_collate(tok))


def _predict(model, loader):
    """Returns (gold, predicted, probability matrix)."""
    model.eval(); gold, probs = [], []
    with torch.no_grad():
        for b in loader:
            y = b.pop("labels")
            b = {k: v.to(DEVICE) for k, v in b.items()}
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(**b).logits
            probs.append(torch.softmax(logits.float(), dim=-1).cpu())
            gold += y.tolist()
    p = torch.cat(probs).numpy() if probs else np.zeros((0, 1))
    return gold, p.argmax(1).tolist(), p


def _macro_f1(gold, pred, n_classes):
    """Fixed denominator. Without labels=, sklearn averages over the labels that
    happen to appear in gold OR pred — which changes epoch to epoch, so scores
    aren't comparable and 'best epoch' picks the wrong checkpoint."""
    return f1_score(gold, pred, labels=list(range(n_classes)),
                    average="macro", zero_division=0)


def _train_one(model_name, seed, tr, va, cmap, epochs):
    torch.manual_seed(seed); np.random.seed(seed)
    n_classes = len(cmap)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=n_classes).to(DEVICE)

    tl, vl = _loader(tr, cmap, tok, shuffle=True), _loader(va, cmap, tok)
    counts = tr["label"].map(cmap).value_counts()
    w = torch.tensor([len(tr) / (n_classes * max(int(counts.get(i, 0)), 1))
                      for i in range(n_classes)], dtype=torch.float).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = max(len(tl) * epochs, 1)
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP_FRAC * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_f1, best_state, best_ep = -1.0, None, 0
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time()
        for b in tl:
            y = b.pop("labels").to(DEVICE)
            b = {k: v.to(DEVICE) for k, v in b.items()}
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = crit(model(**b).logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            scaler.step(opt); scaler.update(); sch.step()

        g, p, _ = _predict(model, vl)                     # selection on VAL only
        f1 = _macro_f1(g, p, n_classes)
        if f1 > best_f1:
            best_f1, best_ep = f1, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"    epoch {ep}: val macro-F1 = {f1:.3f}  ({time.time()-t0:.0f}s)"
              + ("  <- best" if ep == best_ep else ""))
    del model; torch.cuda.empty_cache()
    return best_f1, best_ep, best_state, tok


def _sweep_threshold(model, tok, va, cmap):
    """The app doesn't take argmax — it gates on P(fallacy) >= threshold. Config
    hardcodes 0.70, which was never measured. Find the threshold that actually
    maximises macro-F1 on VAL and ship it with the model."""
    fi = cmap["fallacy"]
    gold, _, probs = _predict(model, _loader(va, cmap, tok))
    gold = np.array(gold)
    best = (0.5, -1.0, None)
    for th in np.arange(0.30, 0.96, 0.01):
        pred = np.where(probs[:, fi] >= th, fi, 1 - fi)
        f1 = _macro_f1(gold.tolist(), pred.tolist(), len(cmap))
        if f1 > best[1]:
            best = (round(float(th), 2), f1, pred)
    th, f1, pred = best
    neg = 1 - fi
    fp = int(((gold == neg) & (pred == fi)).sum()); tn = int(((gold == neg) & (pred == neg)).sum())
    print(f"\n  tuned detector threshold = {th:.2f}  (val macro-F1 {f1:.3f}, "
          f"false alarms {fp}/{tn+fp} = {fp/max(tn+fp,1):.0%})")
    return th


def train_stage(name, tr, va, te, out_dir, epochs, tune_threshold=False):
    """Sweep encoders x seeds, keep the best on VAL, report the honest score on TEST."""
    classes = sorted(tr["label"].unique())
    cmap = {c: i for i, c in enumerate(classes)}
    va, te = va[va["label"].isin(cmap)], te[te["label"].isin(cmap)]

    print(f"\n=== {name}: {len(tr)} train / {len(va)} val / {len(te)} test | {len(classes)} classes ===")
    best = {"f1": -1.0}
    for model_name in MODEL_CANDIDATES:
        for seed in SEEDS:
            print(f"\n  --- {model_name}  seed={seed} ---")
            f1, ep, state, tok = _train_one(model_name, seed, tr, va, cmap, epochs)
            if f1 > best["f1"]:
                best = {"f1": f1, "model": model_name, "seed": seed, "epoch": ep,
                        "state": state, "tok": tok}
            print(f"    -> best val macro-F1 {f1:.3f} at epoch {ep}")

    print(f"\n  WINNER: {best['model']} seed={best['seed']} epoch={best['epoch']} "
          f"(val macro-F1 {best['f1']:.3f})")

    model = AutoModelForSequenceClassification.from_pretrained(
        best["model"], num_labels=len(classes))
    model.load_state_dict(best["state"])
    model.to(DEVICE).eval()
    tok = best["tok"]

    threshold = _sweep_threshold(model, tok, va, cmap) if tune_threshold else None

    # ---- the honest number: TEST set, never used for any decision ----
    gold, preds, probs = _predict(model, _loader(te, cmap, tok))
    if threshold is not None:                     # score the way the app will run it
        fi = cmap["fallacy"]
        preds = np.where(probs[:, fi] >= threshold, fi, 1 - fi).tolist()
    print(f"\n  >> HELD-OUT REAL-WORLD TEST <<")
    print(f"  test macro-F1 = {_macro_f1(gold, preds, len(classes)):.3f}")
    print(classification_report(gold, preds, labels=range(len(classes)),
                                target_names=classes, zero_division=0, digits=2))
    if "not_fallacy" in cmap:
        cm = confusion_matrix(gold, preds, labels=[cmap["not_fallacy"], cmap["fallacy"]])
        tn, fp = cm[0]
        print(f"  false-alarm rate on clean prose: {fp/max(tn+fp,1):.0%}  ({fp}/{tn+fp})")
        print("  (old BERT detector was 62%; TF-IDF v2 was 24%)")

    out = os.path.join(OUT_DIR, out_dir); os.makedirs(out, exist_ok=True)
    model.save_pretrained(out); tok.save_pretrained(out)
    meta = {"classes": classes, "version": DATA_VERSION, "base_model": best["model"]}
    if threshold is not None:
        meta["detect_threshold"] = threshold
    json.dump(meta, open(os.path.join(out, "classes.json"), "w"), indent=2)
    print(f"  -> saved {out_dir}  ({meta})")
    del model; torch.cuda.empty_cache()
    return classes, threshold

### Stage 1 — Detector (the one that was broken)

In [ ]:
det_classes, DET_THRESHOLD = train_stage(
    "DETECTOR", det_train, det_val, det_test, "detector_distilbert",
    DET_EPOCHS, tune_threshold=True)

### Stage 2 — Typer

In [ ]:
typ_classes, _ = train_stage(
    "TYPER", typ_train, typ_val, typ_test, "typer_distilbert", TYP_EPOCHS)

## 4. Write pipeline metadata

In [ ]:
meta = {"detector_model": "bert", "typer_model": "bert",
        "detector_classes": det_classes, "typer_classes": typ_classes,
        "max_len": MAX_LEN, "data_version": DATA_VERSION,
        "detect_threshold": DET_THRESHOLD}
json.dump(meta, open(os.path.join(OUT_DIR, "pipeline.json"), "w"), indent=2)
print(json.dumps(meta, indent=2))
print("\ncontents:", sorted(os.listdir(OUT_DIR)))
print(f"\nThe app reads detect_threshold from detector_distilbert/classes.json.")
print(f"If it ignores it (older classifier.py), set:  $env:FALLACY_DETECT_THRESHOLD = \"{DET_THRESHOLD}\"")

## 5. Sanity check — try it on real debate lines

The old model called *"Thank you both."* a fallacy with 0.99 confidence. The new one should
be quiet on the clean lines and confident on the real fallacies.

In [ ]:
det_dir = os.path.join(OUT_DIR, "detector_distilbert")
typ_dir = os.path.join(OUT_DIR, "typer_distilbert")
det_tok = AutoTokenizer.from_pretrained(det_dir)
det_mdl = AutoModelForSequenceClassification.from_pretrained(det_dir).to(DEVICE).eval()
typ_tok = AutoTokenizer.from_pretrained(typ_dir)
typ_mdl = AutoModelForSequenceClassification.from_pretrained(typ_dir).to(DEVICE).eval()

TYPE_THRESHOLD = 0.50    # same gate the app applies (config.TYPE_THRESHOLD)

def probs(tok, mdl, classes, text):
    enc = tok(text, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    enc.pop("token_type_ids", None)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        p = torch.softmax(mdl(**enc).logits[0].float(), dim=-1)
    return {c: float(p[i]) for i, c in enumerate(classes)}

samples = [
    ("CLEAN", "Thank you both."),
    ("CLEAN", "The Congressional Budget Office found the reform would reduce the deficit by 200 billion over ten years."),
    ("CLEAN", "South Korea and France built fleets on schedule and at cost by standardizing designs."),
    ("FALLACY (false dilemma)", "Either we build firm clean capacity, or we keep burning fossil fuels to back up the intermittent stuff."),
    ("FALLACY (ad hominem)",    "Interesting that the person telling us to bet the grid on nuclear spent six years consulting for a reactor vendor."),
    ("FALLACY (hasty gen.)",    "Two people on my street died of cancer within a few years of each other, and I don't think I'm wrong to generalize from it."),
    ("FALLACY (authority)",     "Even a Nobel laureate has said nuclear is essential, and when a mind like that weighs in, the debate is largely settled."),
]

print(f"gates: P(fallacy) >= {DET_THRESHOLD}  AND  P(type) >= {TYPE_THRESHOLD}\n")
print(f"{'expected':26s} {'P(fallacy)':>11}  {'flagged':>7}  type (confidence = product)")
print("-" * 96)
for tag, s in samples:
    pf = probs(det_tok, det_mdl, det_classes, s).get("fallacy", 0.0)
    tp = probs(typ_tok, typ_mdl, typ_classes, s)
    top = max(tp, key=tp.get)
    flagged = pf >= DET_THRESHOLD and tp[top] >= TYPE_THRESHOLD
    shown = f"{top} ({pf*tp[top]:.2f})" if flagged else f"-   [best: {top} {tp[top]:.2f}]"
    print(f"{tag:26s} {pf:11.2f}  {'YES' if flagged else 'no':>7}  {shown}")

print("\nWant: 'no' on all three CLEAN lines, 'YES' on all four FALLACY lines.")
print("The false dilemma is the one the old detector missed (it rated it 0.41).")

## 6. Download the models — **one at a time**

⚠️ **Do not zip both models together.** A combined zip is ~500 MB and Colab's
`files.download()` silently truncates large files. A zip is written sequentially, so a cut
partway through leaves you with a corrupt first model and a *completely missing* second one
— which looks like "training didn't finish" but is really just a failed download.

So we make **two separate zips (~250 MB each)** and print the exact byte size of each.
**Check the downloaded file matches the size printed here** before using it.

In [ ]:
import shutil, os

sizes = {}
for stage in ["detector_distilbert", "typer_distilbert"]:
    src = os.path.join(OUT_DIR, stage)
    assert os.path.exists(os.path.join(src, "model.safetensors")), f"{stage} did not train!"
    z = shutil.make_archive(f"/content/{stage}", "zip", root_dir=OUT_DIR, base_dir=stage)
    sizes[z] = os.path.getsize(z)
    print(f"{os.path.basename(z):28s} {sizes[z]:,} bytes  ({sizes[z]/1e6:.0f} MB)")

# pipeline.json is tiny — bundle it with nothing else
with open(os.path.join(OUT_DIR, "pipeline.json")) as fh:
    print("\npipeline.json:\n", fh.read())

print("\n>>> VERIFY each downloaded .zip matches the byte size above. If it doesn't,")
print(">>> re-run this cell (or use the Google Drive route in the next cell).")

In [ ]:
from google.colab import files
for stage in ["detector_distilbert", "typer_distilbert"]:
    files.download(f"/content/{stage}.zip")
files.download(os.path.join(OUT_DIR, "pipeline.json"))

### More reliable: copy to Google Drive instead

If a browser download truncates again, push the models to Drive and download them from
there — Drive supports resuming, so large files arrive intact.

In [ ]:
# OPTIONAL — run only if the direct downloads keep truncating.
from google.colab import drive
drive.mount("/content/drive")

dest = "/content/drive/MyDrive/fallacy_models_v2"
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(OUT_DIR, dest)
print("copied to Drive:", dest)
for root, _, fs in os.walk(dest):
    for f in fs:
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, dest):48s} {os.path.getsize(p):,} bytes")

---
## After downloading

```powershell
# unzip detector_distilbert.zip and typer_distilbert.zip into
#   fallacysuspect\models\two_stage\   (overwrite the old folders)
# and drop pipeline.json in beside them

cd C:\Users\Absol\OneDrive\Documents\GitHub\PortFol\fallacysuspect
python scripts\check_models.py     # catches a truncated safetensors before the app sees it
python -m fallacy_warn serve
```

Paste the nuclear debate. The **false dilemma** ("Either we build firm clean capacity, or…")
should now be caught — the old detector rated it 0.41 and missed it entirely.

**The threshold now travels with the model.** `classes.json` carries the `detect_threshold`
that was measured on held-out data, and `classifier.py` reads it. You do not need to edit
`config.py`. To override anyway:

```powershell
$env:FALLACY_DETECT_THRESHOLD = "0.80"   # stricter -> fewer flags
$env:FALLACY_TYPE_THRESHOLD   = "0.55"
```

### Heads-up: these weights cannot go in the GitHub repo as-is
GitHub rejects any single file over 100 MB, and `model.safetensors` is 265 MB (distilbert)
or 330 MB (distilroberta). Options: Git LFS, or host them on the Hugging Face Hub and
download on first run. The TF-IDF joblibs (a few MB) remain the committable fallback.